# 02 — Preprocessing & Label Generation
### NutriFit-AI

This notebook implements **Implementation Plan §2.2** — the key methodological
decision of the project.

No public dataset contains "this person's correct daily calorie target given
their fitness goal". That label is **constructed** here from established
sports-nutrition formulas, transparently and reproducibly.

**Runtime:** CPU. Takes well under a minute.

In [ ]:
# ============================================================
# SETUP - run this first in every notebook
# ============================================================
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/NutriFit-AI")
    if not PROJECT.exists():
        raise FileNotFoundError(
            f"{PROJECT} not found.\n"
            "Upload the whole NutriFit-AI folder to the ROOT of your Google Drive "
            "(My Drive/NutriFit-AI), then re-run this cell."
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "scikit-learn>=1.4", "pandas>=2.1", "joblib>=1.3", "seaborn>=0.13"],
        check=False,
    )
else:
    PROJECT = Path.cwd()
    while not (PROJECT / "ml" / "nutrifit").exists() and PROJECT != PROJECT.parent:
        PROJECT = PROJECT.parent

sys.path.insert(0, str(PROJECT / "ml"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nutrifit
from nutrifit import config, data, foods, labels, nutrition, planner, preprocessing, recommender, training

for directory in (config.PROCESSED_DIR, config.ARTIFACTS_DIR, config.FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

print(f"nutrifit  {nutrifit.__version__}")
print(f"project   {PROJECT}")
print(f"data/raw  {config.RAW_DIR}")
print(f"figures   {config.FIGURES_DIR}")
print(f"in colab  {IN_COLAB}")

In [ ]:
def savefig(name):
    """Save the current figure into reports/figures/ for the dissertation."""
    path = config.FIGURES_DIR / f"{name}.png"
    plt.savefig(path)
    print(f"saved {path}")

## 1. Load and clean

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)-8s %(message)s")

USE_DEMO = False

if USE_DEMO:
    from nutrifit.demo import make_demo_gym_dataset, make_demo_food_dataset
    from nutrifit.schema import (GYM_ALIASES, GYM_REQUIRED, FOOD_ALIASES,
                                 FOOD_REQUIRED, normalise_columns)
    gym = normalise_columns(make_demo_gym_dataset(), GYM_ALIASES, GYM_REQUIRED, "demo")
    gym["height_cm"] = gym["height_m"] * 100
    gym["bmi"] = gym["weight_kg"] / gym["height_m"] ** 2
    food_log = normalise_columns(make_demo_food_dataset(), FOOD_ALIASES, FOOD_REQUIRED, "demo")
    food_log["meal_type"] = food_log["meal_type"].str.lower()
    print("*** SYNTHETIC DEMO DATA - not reportable ***")
else:
    gym = data.load_gym_members()
    food_log = data.load_food_dataset()

print(f"gym {gym.shape} | food log {food_log.shape}")

## 2. The label-generation strategy

Four steps, each grounded in published literature:

| Step | Method | Source |
|---|---|---|
| Fitness goal | Sampled from a softmax over BMI / body-fat / experience | plausible synthetic assignment |
| BMR | **Katch–McArdle** where body fat is measured, else **Mifflin–St Jeor** | Katch & McArdle (1996); Mifflin et al. (1990) |
| TDEE | BMR × activity multiplier (PAL) | FAO/WHO/UNU (2004) |
| Calorie target | TDEE × goal factor sampled within published range | ISSN position stand |
| Protein target | bodyweight × g/kg coefficient sampled within published range | Jäger et al. (2017); Helms et al. (2014) |

Three deliberate sources of variation prevent the models from trivially
memorising a deterministic formula:

1. **Probabilistic goal assignment** — correlated with body composition, not uniform.
2. **Per-person coefficient jitter** — published guidance gives *ranges*, not point values.
3. **Gaussian residual noise** — 3 % CV on calories, 2.5 % on protein.

In [ ]:
labelled = labels.generate_labels(gym, seed=config.RANDOM_SEED)
print(f"Rows: {len(labelled)}")
labelled[["age", "gender", "weight_kg", "height_cm", "bmi", "body_fat_pct",
          "fitness_goal", "activity_level", "bmr", "tdee",
          "calorie_target", "protein_target"]].head(10)

### 2.1 Goal distribution and its correlation with body composition

Goals are *not* assigned uniformly at random — a user at high body fat is more
likely pursuing fat loss. This induces the realistic feature–label correlations
an ML model is supposed to discover.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
order = list(nutrition.GOALS)
sns.countplot(data=labelled, x="fitness_goal", order=order, ax=axes[0])
axes[0].set_title("Assigned fitness goal")
sns.boxplot(data=labelled, x="fitness_goal", y="bmi", order=order, ax=axes[1])
axes[1].set_title("BMI by goal")
sns.boxplot(data=labelled, x="fitness_goal", y="body_fat_pct", order=order, ax=axes[2])
axes[2].set_title("Body fat % by goal")
plt.tight_layout()
savefig("labels_goal_assignment")
plt.show()

display(labelled.groupby("fitness_goal")[["bmi", "body_fat_pct", "weight_kg"]].mean().round(2))

### 2.2 The constructed targets

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.histplot(labelled["calorie_target"], bins=35, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Daily calorie target (kcal)")
sns.histplot(labelled["protein_target"], bins=35, kde=True, ax=axes[0, 1], color="darkorange")
axes[0, 1].set_title("Daily protein target (g)")
sns.boxplot(data=labelled, x="fitness_goal", y="calorie_target", order=order, ax=axes[1, 0])
axes[1, 0].set_title("Calorie target by goal")
sns.boxplot(data=labelled, x="fitness_goal", y="protein_target", order=order, ax=axes[1, 1])
axes[1, 1].set_title("Protein target by goal")
plt.tight_layout()
savefig("labels_target_distributions")
plt.show()

display(labelled[["bmr", "tdee", "calorie_target", "protein_target"]].describe().T.round(1))

### 2.3 Sanity check — goal ordering

Normalised against each user's own TDEE, the ordering must be
`fat_loss < maintenance < muscle_gain`. If this fails, the label generator is broken.

In [ ]:
ratio = (labelled["calorie_target"] / labelled["tdee"]).groupby(labelled["fitness_goal"]).mean()
display(ratio.round(4).rename("calorie_target / TDEE").to_frame())
assert ratio["fat_loss"] < ratio["maintenance"] < ratio["muscle_gain"], "goal ordering violated"
print("PASS - deficit < maintenance < surplus")

protein_per_kg = (labelled["protein_target"] / labelled["weight_kg"]).groupby(labelled["fitness_goal"]).mean()
display(protein_per_kg.round(3).rename("protein g/kg").to_frame())

### 2.4 The theoretical R² ceiling

The injected noise imposes a hard upper bound on achievable R²:

$$R^2_{max} = 1 - \frac{Var(noise)}{Var(target)}$$

Reporting this alongside the observed model R² shows how much of the remaining
error is **irreducible**. It also pre-empts the obvious examiner question:
*"why isn't your R² higher?"*

In [ ]:
for target in ("calorie_target", "protein_target"):
    ceiling = labels.theoretical_r2_ceiling(labelled, target)
    print(f"{target:16s} theoretical R2 ceiling = {ceiling:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, target in zip(axes, ("calorie_target", "protein_target")):
    ax.scatter(labelled[f"{target}_clean"], labelled[target], s=8, alpha=0.4)
    lo = min(labelled[f"{target}_clean"].min(), labelled[target].min())
    hi = max(labelled[f"{target}_clean"].max(), labelled[target].max())
    ax.plot([lo, hi], [lo, hi], "r--", lw=1.5, label="noise-free")
    ax.set_xlabel("formula value"); ax.set_ylabel("label (with noise)")
    ax.set_title(target); ax.legend()
plt.tight_layout()
savefig("labels_noise_injection")
plt.show()

## 3. The preprocessing pipeline (Implementation Plan §3.2)

All preprocessing lives inside one `ColumnTransformer` that is fitted **inside**
the same `Pipeline` as the estimator. Two consequences:

* the exported `.pkl` contains preprocessing **and** model, so the FastAPI
  service applies byte-identical transforms — no train/serve skew;
* cross-validation re-fits the imputer/scaler on each training fold only, so no
  information leaks through the scaler statistics.

**Note on the feature contract:** the models see only *raw* user attributes.
Derived physiology (`bmr`, `activity_multiplier`, `tdee`) is deliberately
excluded — handing the model the multiplier would give away the answer.

In [ ]:
print("Numeric  :", preprocessing.NUMERIC_FEATURES)
print("Nominal  :", preprocessing.NOMINAL_FEATURES)
print("Ordinal  :", preprocessing.ORDINAL_FEATURES)
print("Targets  :", preprocessing.TARGET_COLUMNS)

from sklearn.linear_model import LinearRegression
demo_pipe = preprocessing.build_pipeline(LinearRegression())
X = preprocessing.select_features(labelled)
demo_pipe.fit(X, labelled["calorie_target"])

print(f"\nRaw features      : {X.shape[1]}")
print(f"After transform   : {len(preprocessing.get_feature_names(demo_pipe))}")
print(preprocessing.get_feature_names(demo_pipe))

### 3.1 Robustness checks the pipeline must pass

In [ ]:
row = X.iloc[[0]].copy()

missing = row.copy(); missing.loc[:, "weight_kg"] = np.nan; missing.loc[:, "gender"] = None
print(f"missing values    -> {demo_pipe.predict(missing)[0]:.1f} kcal")

unseen = row.copy(); unseen.loc[:, "activity_level"] = "hyperactive"; unseen.loc[:, "gender"] = "Other"
print(f"unseen categories -> {demo_pipe.predict(unseen)[0]:.1f} kcal")

shuffled = row[list(reversed(preprocessing.FEATURE_COLUMNS))]
print(f"shuffled columns  -> {demo_pipe.predict(shuffled)[0]:.1f} kcal "
      f"(must equal {demo_pipe.predict(row)[0]:.1f})")

extreme = row.copy(); extreme.loc[:, "weight_kg"] = 700.0
print(f"absurd weight     -> {demo_pipe.predict(extreme)[0]:.1f} kcal (IQR-clipped)")

## 4. Build the food catalogue

In [ ]:
# USDA FoodData Central enrichment.
#
# Measured effect on the eight-week plan (worst weekly macro error across
# all three goals, 8 weeks, seed 42):
#     Kaggle only (489 items) : 3.80 %, 2 repair passes needed
#     + USDA      (593 items) : 2.88 %, 0 repair passes needed
#
# The USDA rows are per 100 g while the Kaggle rows are per serving. That
# mixing of measurement bases was expected to hurt, but the extra whole foods
# are protein-dense, and protein is the binding constraint - so it measurably
# helps. Set to False to compare for yourself; it is a good ablation to report.
USE_USDA = True

usda = data.load_usda_foundation() if (USE_USDA and not USE_DEMO) else None
if usda is not None:
    print('USDA enrichment ON:', len(usda), 'candidate foods loaded')
else:
    print('USDA enrichment OFF - catalogue built from the Kaggle dataset only')
catalogue = foods.build_catalogue(food_log, usda=usda)
health = foods.catalogue_health_report(catalogue)
display(health)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
slot_order = [m for m in nutrition.MEAL_SLOTS if m in set(catalogue["meal_type"])]
sns.boxplot(data=catalogue, x="meal_type", y="calories", order=slot_order, ax=axes[0])
axes[0].set_title("Catalogue calories per item, by slot")
sns.boxplot(data=catalogue, x="meal_type", y="protein_density", order=slot_order, ax=axes[1])
axes[1].set_title("Protein density (g per 100 kcal)")
plt.tight_layout()
savefig("catalogue_macro_profile")
plt.show()

## 5. Persist processed artefacts

In [ ]:
labelled.to_csv(config.USERS_PROCESSED, index=False)
catalogue.to_csv(config.FOODS_PROCESSED, index=False)
config.FOODS_SEED_SQL.write_text(foods.to_sql_seed(catalogue), encoding="utf-8")

print(f"users     -> {config.USERS_PROCESSED}  ({len(labelled)} rows)")
print(f"catalogue -> {config.FOODS_PROCESSED}  ({len(catalogue)} items)")
print(f"seed sql  -> {config.FOODS_SEED_SQL}")

## 6. What to write in the Methodology chapter

> *The models are trained to approximate an established physiological
> relationship from raw user features, and are evaluated on their ability to
> generalise it. Because BMR is estimated with Katch–McArdle where body
> composition is known, and because the goal-specific factors are applied
> multiplicatively, the target is not an additive function of the inputs —
> which is precisely the structure Random Forest is expected to capture and
> Linear Regression is not.*

Notebook 03 tests that expectation. **The result is not what the proposal
predicted** — see notebook 04 for the analysis and the evidence.

**Next:** `03_model_training.ipynb`